# Genex Interview + Activity Brain — v2

This notebook is the **cleaned working notebook** for the current Genex 0–5 developmental interview and activity-planning engine.

It supports **two interview modes** that share the same downstream planning pipeline:

## Mode A — Live parent mode
You answer the milestone questions yourself, one by one, as if you were the parent.

## Mode B — Synthetic parent mode
Gemini simulates the parent answers one question at a time from the child profile, while the Genex interview engine still selects the questions and computes the developmental profile.

After either mode, the notebook runs the **same shared downstream steps**:
1. delay estimation by domain  
2. milestone question interview by category  
3. developmental-age estimate by category  
4. category activity-bank generation  
5. weekly schedule generation  
6. case summary table  
7. optional advisor-report export

---

## What this notebook is for
Use this notebook when you want to:
- test the milestone interview logic
- run one real child case live
- run a set of synthetic benchmark cases quickly
- generate advisor-review case packets
- produce a weekly parent-facing schedule from the same pipeline

---

## What this notebook is **not** for
This notebook is **not** the place for:
- HPO raw-data processing
- HPO table building
- PubMed / RAG experiments
- Bright Futures parsing
- external literature retrieval benchmarking

Those should live in separate notebooks.

---

## Core design principle
There is **one main planning engine** and **two input modes**.

That means:
- live mode and synthetic mode only differ in **how answers are obtained**
- everything after the Q&A is shared
- export/report logic is purely downstream and does not rerun inference

---

## Model Architecture 

- Genex local logic picks the questions
- OpenAI helps with delay/activity generation
- Gemini answers like the parent

---

## Inputs
### Live mode
- child name
- age in months
- diagnosis / condition
- parent concerns
- daily time available
- milestone answers entered by you

### Synthetic mode
- case profile from `df_cases`
- daily time available
- Gemini simulates milestone answers from the child profile

---

## Outputs
This notebook can produce:
- screen-based category summaries
- a structured `summary_df`
- weekly schedule display
- one-page-per-case PDF report
- advisor rating CSV / XLSX
- JSON payloads for case review

---

## Security note
Do **not** hardcode API keys into the notebook.
Use environment variables or temporary notebook cells that you do not save.

Expected environment variables:
- `OPENAI_API_KEY`
- `GEMINI_API_KEY`

---

## Recommended execution order
1. Install cell  
2. Imports + environment setup  
3. CDC loading  
4. Core engine cells  
5. Choose one mode:
   - run one live case
   - run one synthetic case
   - run multiple cases and export advisor packet

## Notebook sections

1. Install
2. Imports + environment setup
3. Config + CDC loading
4. Core milestone engine
5. Q&A modes
   - live parent
   - Gemini synthetic parent
6. Shared planning engine
7. Main runners
8. Test cases
9. Advisor-report export
10. Example run cells

In [1]:
# Install once per notebook kernel if needed
%pip install -U pandas openpyxl python-dotenv openai google-genai reportlab


[notice] A new release of pip is available: 26.0.1 -> 26.1
[notice] To update, run: /opt/homebrew/opt/python@3.11/bin/python3.11 -m pip install --upgrade pip
Note: you may need to restart the kernel to use updated packages.


In [2]:
# ------------------------------------------------------------
# Imports + environment setup
# ------------------------------------------------------------
import os
import time
import re
import json
from pathlib import Path
from datetime import datetime
from typing import Any, Dict, List, Optional

import pandas as pd
from dotenv import load_dotenv
from IPython.display import display, Markdown

try:
    from openai import OpenAI
except ImportError:
    OpenAI = None

try:
    from google import genai
except ImportError:
    genai = None

from reportlab.lib.pagesizes import landscape, LETTER
from reportlab.lib.units import inch
from reportlab.pdfgen import canvas
from reportlab.pdfbase.pdfmetrics import stringWidth

load_dotenv()

PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
OUTPUT_DIR = PROJECT_ROOT / "outputs" / "genex_interview_activity_v2"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

RUN_STAMP = datetime.now().strftime("%Y%m%d_%H%M%S")

OPENAI_API_KEY = os.environ.get("OPENAI_API_KEY")
GEMINI_API_KEY = os.environ.get("GEMINI_API_KEY")

if OpenAI is None:
    print("Warning: openai package is not available yet.")
if not OPENAI_API_KEY:
    print("Warning: OPENAI_API_KEY is not visible. OpenAI-backed steps will use fallbacks where possible.")

if genai is None:
    print("Warning: google-genai package is not available yet.")
if not GEMINI_API_KEY:
    print("Warning: GEMINI_API_KEY is not visible. Synthetic Gemini-parent mode will not work until it is set.")

gemini_client = None
if genai is not None and GEMINI_API_KEY:
    gemini_client = genai.Client(api_key=GEMINI_API_KEY)
    print("Gemini client ready.")

# Optional temporary setup if you need to set keys inside the notebook.
# Do not save real keys in the notebook file.
#
# import os
# os.environ["OPENAI_API_KEY"] = "your_key_here"
# os.environ["GEMINI_API_KEY"] = "your_key_here"

In [ ]:
import os
from google import genai

os.environ["GEMINI_API_KEY"] = ""

print("GEMINI_API_KEY visible:", bool(os.environ.get("GEMINI_API_KEY")))

gemini_client = genai.Client(api_key=os.environ["GEMINI_API_KEY"])
print("gemini_client is None:", gemini_client is None)

GEMINI_API_KEY visible: True
gemini_client is None: False


In [4]:
from openai import OpenAI

OPENAI_API_KEY = os.environ.get("OPENAI_API_KEY")
if not OPENAI_API_KEY:
    raise ValueError("OPENAI_API_KEY is missing.")

openai_client = OpenAI(api_key=OPENAI_API_KEY)
print("OpenAI client ready.")

OpenAI client ready.


In [5]:
# ------------------------------------------------------------
# Config
# ------------------------------------------------------------
DOMAIN_CONFIG = {
    "movement_and_physical": {
        "display": "Movement / Physical",
        "short": "motor",
    },
    "social_and_emotional": {
        "display": "Social / Emotional",
        "short": "social_emotional",
    },
    "language_and_communication": {
        "display": "Language / Communication",
        "short": "language_communication",
    },
    "cognitive": {
        "display": "Cognitive / Adaptive",
        "short": "cognitive",
    },
}

ALIAS_TO_CATEGORY = {
    "movement and physical": "movement_and_physical",
    "physical": "movement_and_physical",
    "motor": "movement_and_physical",
    "gross motor": "movement_and_physical",

    "social and emotional": "social_and_emotional",
    "social and emotial": "social_and_emotional",
    "social_emotional": "social_and_emotional",
    "social": "social_and_emotional",

    "language and communication": "language_and_communication",
    "language": "language_and_communication",
    "speech": "language_and_communication",
    "speech and language": "language_and_communication",

    "cognitive": "cognitive",
    "adaptive": "cognitive",
}

ANSWER_SCORES = {
    "yes": 1.0,
    "sometimes": 0.4,
    "with_help": 0.2,
    "no": 0.0,
    "not_sure": 0.1,
}

VALID_ANSWERS = set(ANSWER_SCORES.keys())
VALID_PARENT_ANSWERS = {"yes", "sometimes", "with_help", "no", "not_sure"}

In [6]:
# ------------------------------------------------------------
# CDC loading + normalization
# ------------------------------------------------------------
def find_cdc_file(filename: str = "milestone-cdc-table.xlsx") -> Path:
    search_roots = [Path.cwd(), Path.cwd().parent, Path.cwd().parent.parent]
    for root in search_roots:
        candidate = root / filename
        if candidate.exists():
            return candidate.resolve()
    for root in search_roots:
        if root.exists():
            matches = list(root.rglob(filename))
            if matches:
                return matches[0].resolve()
    raise FileNotFoundError(
        f"Could not find {filename}. Put it in the repo root or notebooks parent folder."
    )

def load_cdc_table(path: Optional[str] = None):
    path = Path(path) if path else find_cdc_file()
    df = pd.read_excel(path)

    df.columns = [str(c).strip().lower() for c in df.columns]
    df = df.rename(columns={"category ": "category", "milestone ": "milestone"})
    df["category"] = df["category"].astype(str).str.strip().str.lower()
    df["milestone"] = df["milestone"].astype(str).str.strip()
    df["months"] = pd.to_numeric(df["months"], errors="coerce")
    df = df.dropna(subset=["months", "category", "milestone"]).copy()
    df["months"] = df["months"].astype(int)
    df["category_key"] = df["category"].map(lambda x: ALIAS_TO_CATEGORY.get(x, x.replace(" ", "_")))
    df["question_id"] = [
        f"{row.category_key}_{row.months}_{i}"
        for i, row in enumerate(df.itertuples(), start=1)
    ]
    return df, path

cdc_df, cdc_path = load_cdc_table()
CDC_AGES = sorted(cdc_df["months"].dropna().unique().tolist())

print("Loaded CDC file:", cdc_path)
print("CDC ages:", CDC_AGES)
print("Rows:", len(cdc_df))

def get_category_questions(category_key: str, min_months: int, max_months: int) -> pd.DataFrame:
    subset = cdc_df[
        (cdc_df["category_key"] == category_key)
        & (cdc_df["months"] >= min_months)
        & (cdc_df["months"] <= max_months)
    ].sort_values(["months", "milestone"])
    return subset.copy()

display(cdc_df.head())

Loaded CDC file: /Users/sara/Projects/Genex/milestone-cdc-table.xlsx
CDC ages: [2, 4, 6, 9, 12, 15, 18, 24, 30, 36, 48, 60]
Rows: 159


,months,category,milestone,category_key,question_id
0,2,social and emotial,calms down when spoken to or picked up,social_and_emotional,social_and_emotional_2_1
1,2,social and emotial,looks at your face,social_and_emotional,social_and_emotional_2_2
2,2,social and emotial,seems happy to see you when you walk up to her,social_and_emotional,social_and_emotional_2_3
3,2,social and emotial,smiles when you talk to or smile at her,social_and_emotional,social_and_emotional_2_4
4,2,language and communication,makes sounds other than crying,language_and_communication,language_and_communication_2_5


In [7]:
# ------------------------------------------------------------
# Core helpers + state
# ------------------------------------------------------------
def normalize_answer(answer_text: str) -> str:
    t = str(answer_text).strip().lower().replace(" ", "_")
    if t in VALID_ANSWERS:
        return t
    if t in {"notsure", "unsure", "maybe"}:
        return "not_sure"
    return "not_sure"

def print_banner(title: str):
    print("\n" + "=" * 100)
    print(title)
    print("=" * 100)

def extract_json_block(text: str) -> dict:
    text = str(text).strip()

    # 1. direct parse
    try:
        return json.loads(text)
    except Exception:
        pass

    # 2. fenced json block
    m = re.search(r"```json\s*(\{.*?\})\s*```", text, flags=re.DOTALL)
    if m:
        try:
            return json.loads(m.group(1))
        except Exception:
            pass

    # 3. any json-like object
    m = re.search(r"(\{.*\})", text, flags=re.DOTALL)
    if m:
        try:
            return json.loads(m.group(1))
        except Exception:
            pass

    # 4. fallback: interpret plain text answer
    lowered = text.lower()

    for label in ["with_help", "not_sure", "sometimes", "yes", "no"]:
        if label in lowered:
            return {
                "answer": label,
                "reason": text[:200],
            }

    # extra forgiving fallbacks
    if "with help" in lowered:
        return {"answer": "with_help", "reason": text[:200]}
    if "not sure" in lowered or "unsure" in lowered or "maybe" in lowered:
        return {"answer": "not_sure", "reason": text[:200]}

    raise ValueError(f"Could not parse JSON from model output: {text[:300]}")

def new_state() -> Dict[str, Any]:
    return {
        "child": {},
        "delay_estimates": {},
        "qna": {},
        "dev_age": {},
        "support_recommendations": {},
        "activity_banks": {},
        "weekly_slot_allocation": {},
        "weekly_schedule": {},
    }

def print_child_reference(state: Dict[str, Any]) -> None:
    child = state["child"]

    print("\n" + "=" * 100)
    print("CHILD REFERENCE CHECK")
    print("=" * 100)
    print(f"Name: {child.get('name', '')}")
    print(f"Chronological age: {child.get('chronological_months', '')} months")
    print(f"Diagnosis / condition: {child.get('diagnosis', '')}")
    print(f"Parent concern: {child.get('concern', '')}")
    print(f"Daily time available: {child.get('daily_time_min', '')} minutes")

def intake_agent_live(state: Dict[str, Any]) -> Dict[str, Any]:
    print_banner("INTAKE AGENT")

    name = input("What is your child's name? ").strip()
    chronological_months = int(
        input("How old is your child in months? (for example: 18, 24, 36, 48, 60) ").strip()
    )
    diagnosis = input("Any diagnosis / condition? (type 'none' if none) ").strip()
    concern = input("What are your main concerns right now? ").strip()
    daily_time_min = int(
        input("About how many minutes per day can you usually spend helping or playing with your child? ").strip()
    )

    state["child"] = {
        "name": name,
        "chronological_months": chronological_months,
        "diagnosis": diagnosis,
        "concern": concern,
        "daily_time_min": daily_time_min,
    }

    return state["child"]

def init_state_from_case(case: Dict[str, Any], daily_time_min: int = 10) -> Dict[str, Any]:
    state = new_state()
    state["child"] = {
        "name": case["child_name"],
        "chronological_months": int(case["age_months"]),
        "diagnosis": case["diagnosis"],
        "concern": case["concerns"],
        "daily_time_min": int(daily_time_min),
    }
    return state

In [8]:
# ------------------------------------------------------------
# Delay estimation
# ------------------------------------------------------------
def estimate_delay_gpt(
    diagnosis: str,
    concern: str,
    chronological_months: int,
    category_key: str,
    model: str = "gpt-4o-mini",
) -> Dict[str, Any]:
    category_display = DOMAIN_CONFIG[category_key]["display"]

    diagnosis_l = (diagnosis or "").lower()
    concern_l = (concern or "").lower()

    fallback_by_category = {
        "movement_and_physical": 3,
        "language_and_communication": 3,
        "social_and_emotional": 6,
        "cognitive": 6,
    }

    domain_keywords = {
        "movement_and_physical": [
            "motor", "movement", "walk", "run", "jump", "balance", "coordination",
            "fine motor", "gross motor", "grasp", "hand", "writing", "stairs", "falls"
        ],
        "language_and_communication": [
            "speech", "language", "talk", "communication", "words", "sentence",
            "understand", "expressive", "receptive", "verbal", "babbling"
        ],
        "social_and_emotional": [
            "social", "peer", "friend", "play", "emotion", "emotional",
            "behavior", "anger", "meltdown", "interaction", "turn taking",
            "regulation", "eye contact", "transitions"
        ],
        "cognitive": [
            "attention", "focus", "concentration", "school", "learning", "routine",
            "executive", "task", "independent", "adaptive", "toilet", "dressing",
            "self-care", "directions"
        ],
    }

    has_domain_signal = any(
        kw in concern_l for kw in domain_keywords.get(category_key, [])
    )

    if OpenAI is None or not os.environ.get("OPENAI_API_KEY"):
        return {
            "delay_months": fallback_by_category.get(category_key, 6),
            "reason": f"Fallback delay estimate used for {category_display} because the OpenAI client is not available.",
            "source": "fallback",
        }

    client = OpenAI()

    prompt = f"""
You are a pediatric developmental delay estimator agent for children ages 0 to 5 years.

Your job is to estimate a SINGLE STARTING DELAY in months for one developmental domain only.

This is NOT a diagnosis.
This is NOT a final developmental age.
This is ONLY a rough starting anchor for question selection.

Definition:
delay_months = chronological age in months - estimated functional developmental age in this specific domain

Child information:
- Chronological age in months: {chronological_months}
- Diagnosis / condition: {diagnosis}
- Parent concern: {concern}
- Domain to estimate: {category_display}

Instructions:
1. Think only about THIS domain, not overall development.
2. If the diagnosis or concern does NOT meaningfully affect this domain, return 0 to 6 months.
3. If this domain IS affected, estimate the child's functional developmental level in this domain, then convert it into delay_months.
4. Be conservative but realistic.
5. Never exceed the child's chronological age.
6. Return only one integer number of months.
7. Return strict JSON only.

Examples:

Example 1:
Child is 60 months old with Down syndrome.
Domain = Language / Communication.
If you think language skills are functioning around 24 months,
return:
{{"delay_months": 36, "reason": "Language development is likely meaningfully affected in this child."}}

Example 2:
Child is 48 months old with autism.
Domain = Social / Emotional.
If you think social interaction skills are functioning around 24 months,
return:
{{"delay_months": 24, "reason": "Social-emotional development is likely meaningfully affected in this child."}}

Example 3:
Child is 60 months old with cerebral palsy.
Domain = Movement / Physical.
If you think motor skills are functioning around 30 months,
return:
{{"delay_months": 30, "reason": "Movement and physical development are likely meaningfully affected in this child."}}

Example 4:
Child is 30 months old with speech delay.
Domain = Language / Communication.
If you think language skills are functioning around 18 months,
return:
{{"delay_months": 12, "reason": "Language development is likely delayed based on the concern."}}

Example 5:
Child is 36 months old with global developmental delay.
Domain = Cognitive / Adaptive.
If you think adaptive / cognitive functioning is around 18 months,
return:
{{"delay_months": 18, "reason": "Cognitive and adaptive development are likely meaningfully affected."}}

Example 6:
Child is 48 months old with autism.
Domain = Movement / Physical.
If there is no meaningful motor concern,
return:
{{"delay_months": 0, "reason": "There is little reason to think this domain is significantly affected."}}

Example 7:
Child is 60 months old with Down syndrome.
Domain = Social / Emotional.
If you think social skills are only mildly affected,
return:
{{"delay_months": 6, "reason": "This domain may be mildly affected but not severely delayed."}}

Required JSON:
{{
  "delay_months": <integer>,
  "reason": "<one short sentence>"
}}
""".strip()

    try:
        resp = client.chat.completions.create(
            model=model,
            temperature=0.2,
            response_format={"type": "json_object"},
            messages=[
                {"role": "system", "content": "You return strict JSON only."},
                {"role": "user", "content": prompt},
            ],
        )

        parsed = json.loads(resp.choices[0].message.content)
        delay_months = int(parsed.get("delay_months", fallback_by_category.get(category_key, 6)))
        delay_months = max(0, min(delay_months, chronological_months))

        if not has_domain_signal and delay_months > 6:
            delay_months = 6

        return {
            "delay_months": delay_months,
            "reason": parsed.get("reason", ""),
            "source": "openai",
        }

    except Exception as e:
        return {
            "delay_months": fallback_by_category.get(category_key, 6),
            "reason": f"Fallback delay estimate used because the OpenAI call failed: {e}",
            "source": "fallback",
        }

def delay_agent_all_categories(state: Dict[str, Any], categories: Optional[List[str]] = None) -> Dict[str, Any]:
    print_banner("DELAY ESTIMATOR AGENT")
    if not state.get("child"):
        raise ValueError("Child profile missing. Run intake first.")

    categories = categories or list(DOMAIN_CONFIG.keys())
    child = state["child"]

    for category_key in categories:
        est = estimate_delay_gpt(
            diagnosis=child["diagnosis"],
            concern=child["concern"],
            chronological_months=child["chronological_months"],
            category_key=category_key,
        )
        state["delay_estimates"][category_key] = est
        print(f"{DOMAIN_CONFIG[category_key]['display']}: {est['delay_months']} month starting delay | {est['reason']}")

    return state["delay_estimates"]

In [9]:
# ------------------------------------------------------------
# Core milestone engine
# ------------------------------------------------------------
def build_milestone_questions(
    state: Dict[str, Any],
    category_key: str,
    window_months: int = 24,
    target_questions_per_band: int = 4,
    max_bands: int = 3,
    max_questions_total: int = 12,
) -> List[Dict[str, Any]]:
    child = state["child"]
    chrono_months = min(child["chronological_months"], 60)
    delay_months = state["delay_estimates"].get(category_key, {}).get("delay_months", 12)

    approx_dev_months = max(2, chrono_months - delay_months)
    min_months = max(2, approx_dev_months - window_months // 2)
    max_months = min(60, approx_dev_months + window_months // 2)

    subset = get_category_questions(category_key, min_months=min_months, max_months=max_months)
    if subset.empty:
        subset = get_category_questions(category_key, min_months=min(CDC_AGES), max_months=max(CDC_AGES))

    available_months = sorted(subset["months"].unique().tolist())

    ordered_months = sorted(
        available_months,
        key=lambda m: (abs(m - approx_dev_months), m)
    )[:max_bands]

    questions = []
    for month in ordered_months:
        month_rows = subset[subset["months"] == month].sort_values("milestone")
        month_rows = month_rows.head(target_questions_per_band)

        for _, row in month_rows.iterrows():
            questions.append({
                "question_id": row["question_id"],
                "months": int(row["months"]),
                "milestone": row["milestone"],
                "question_text": f"Can {child['name']} {row['milestone']} right now? (yes / sometimes / with_help / no / not_sure)",
            })

    return questions[:max_questions_total]

def compute_dev_age_from_answers(
    answers: List[Dict[str, Any]],
    min_yes_confirm: int = 2,
    yes_ratio_confirm: float = 0.70,
) -> int:
    """
    Conservative band-based heuristic.

    A month band is confirmed only if:
    - it has at least `min_yes_confirm` YES answers
    - and YES answers are at least `yes_ratio_confirm` of the answers in that band
    """
    if not answers:
        return 6

    band_summary = {}
    for a in answers:
        month = int(a["months"])
        norm = a["norm_answer"]

        if month not in band_summary:
            band_summary[month] = {
                "total": 0,
                "yes": 0,
                "partial": 0,
                "no": 0,
            }

        band_summary[month]["total"] += 1

        if norm == "yes":
            band_summary[month]["yes"] += 1
        elif norm in {"sometimes", "with_help", "not_sure"}:
            band_summary[month]["partial"] += 1
        else:
            band_summary[month]["no"] += 1

    answered_months = sorted(band_summary.keys())

    confirmed_months = []
    partial_months = []

    for month in answered_months:
        total = band_summary[month]["total"]
        yes_count = band_summary[month]["yes"]
        partial_count = band_summary[month]["partial"]
        yes_ratio = yes_count / total if total > 0 else 0

        if yes_count >= min_yes_confirm and yes_ratio >= yes_ratio_confirm:
            confirmed_months.append(month)
        elif yes_count > 0 or partial_count > 0:
            partial_months.append(month)

    if confirmed_months:
        return int(max(confirmed_months))

    if partial_months:
        anchor = int(min(partial_months))
        lower_candidates = [m for m in answered_months if m < anchor]
        return int(max(lower_candidates)) if lower_candidates else int(anchor)

    return int(min(answered_months))

def summarize_answers_by_band(answers: List[Dict[str, Any]]) -> Dict[int, Dict[str, Any]]:
    band_summary = {}

    for a in answers:
        month = int(a["months"])
        norm = a["norm_answer"]

        if month not in band_summary:
            band_summary[month] = {
                "total": 0,
                "yes": 0,
                "partial": 0,
                "no": 0,
                "items": [],
            }

        band_summary[month]["total"] += 1
        band_summary[month]["items"].append({
            "milestone": a["milestone"],
            "answer": norm,
        })

        if norm == "yes":
            band_summary[month]["yes"] += 1
        elif norm in {"sometimes", "with_help", "not_sure"}:
            band_summary[month]["partial"] += 1
        else:
            band_summary[month]["no"] += 1

    for month in band_summary:
        total = band_summary[month]["total"]
        yes = band_summary[month]["yes"]
        band_summary[month]["yes_ratio"] = round(yes / total, 2) if total else 0.0

    return dict(sorted(band_summary.items()))

In [10]:
# ------------------------------------------------------------
# Q&A mode A — live parent
# ------------------------------------------------------------
def qna_agent_live(
    state: Dict[str, Any],
    category_key: str,
    min_yes_confirm: int = 2,
    yes_ratio_confirm: float = 0.70,
) -> Dict[str, Any]:
    print_banner(f"MILESTONE Q&A AGENT — {DOMAIN_CONFIG[category_key]['display']}")

    if "qna" not in state:
        state["qna"] = {}
    if "dev_age" not in state:
        state["dev_age"] = {}

    questions = build_milestone_questions(
        state,
        category_key,
        window_months=24,
        target_questions_per_band=4,
        max_bands=3,
        max_questions_total=12,
    )

    if not questions:
        raise ValueError(
            f"No milestone questions found for category_key={category_key}. "
            f"Check category mapping and CDC normalization."
        )

    asked = []
    questions_by_month = {}
    for q in questions:
        questions_by_month.setdefault(q["months"], []).append(q)

    for month in sorted(questions_by_month.keys()):
        print(f"\n--- Asking {DOMAIN_CONFIG[category_key]['display']} milestones around {month} months ---")
        for q in questions_by_month[month]:
            ans = input(q["question_text"] + "\n").strip()
            norm = normalize_answer(ans)
            asked.append({
                "question_id": q["question_id"],
                "months": q["months"],
                "milestone": q["milestone"],
                "raw_answer": ans,
                "norm_answer": norm,
                "score": ANSWER_SCORES[norm],
            })

    dev_age = compute_dev_age_from_answers(
        asked,
        min_yes_confirm=min_yes_confirm,
        yes_ratio_confirm=yes_ratio_confirm,
    )

    state["qna"][category_key] = asked
    state["dev_age"][category_key] = dev_age

    chrono = state["child"]["chronological_months"]

    print("\nQuestion / answer transcript:")
    for item in asked:
        print(f"- [{item['months']}m] {item['milestone']} -> {item['norm_answer']}")

    band_summary = summarize_answers_by_band(asked)
    print("\nBand summary:")
    for month, info in band_summary.items():
        print(
            f"  {month}m | total={info['total']} | yes={info['yes']} | "
            f"partial={info['partial']} | no={info['no']} | yes_ratio={info['yes_ratio']}"
        )

    print(
        f"\nEstimated developmental age for {DOMAIN_CONFIG[category_key]['display']}: "
        f"{dev_age} months (chronological age {chrono} months)"
    )

    return {
        "category": category_key,
        "dev_age_months": dev_age,
        "answers": asked,
        "band_summary": band_summary,
    }

In [11]:
# ------------------------------------------------------------
# Q&A mode B — Gemini synthetic parent
# ------------------------------------------------------------
def gemini_parent_answer_one_question(
    child: Dict[str, Any],
    category_display: str,
    question_text: str,
    prior_answers: List[Dict[str, Any]],
    model: str = "gemini-2.5-flash",
    max_retries: int = 3,
) -> Dict[str, Any]:
    if gemini_client is None:
        raise ValueError(
            "Gemini client is not available. Set GEMINI_API_KEY and rerun the environment setup cell."
        )

    if prior_answers:
        prior_text = "\n".join(
            [f"- {x['question']} -> {x['answer']}" for x in prior_answers[-8:]]
        )
    else:
        prior_text = "None yet."

    prompt = f"""
You are simulating a REAL parent answering developmental milestone questions about their child.

Rules:
- Answer ONLY from the child profile below.
- Be internally consistent across questions.
- Do NOT answer like a clinician or therapist.
- Answer like a parent who knows the child in daily life.
- Choose ONLY one answer:
  yes
  sometimes
  with_help
  no
  not_sure

Child profile:
- Name: {child.get('name')}
- Age: {child.get('chronological_months')} months
- Diagnosis / condition: {child.get('diagnosis')}
- Main concern: {child.get('concern')}

Current developmental category:
- {category_display}

Previous answers in this same interview:
{prior_text}

Question:
{question_text}

Return strict JSON only:
{{
  "answer": "yes|sometimes|with_help|no|not_sure",
  "reason": "one short parent-style reason"
}}
""".strip()

    last_error = None

    for attempt in range(max_retries):
        try:
            response = gemini_client.models.generate_content(
                model=model,
                contents=prompt,
            )

            raw_text = response.text if hasattr(response, "text") else str(response)
            parsed = extract_json_block(raw_text)

            answer = str(parsed.get("answer", "")).strip().lower().replace(" ", "_")
            if answer not in VALID_PARENT_ANSWERS:
                answer = "not_sure"

            return {
                "answer": answer,
                "reason": str(parsed.get("reason", "")).strip() or raw_text[:200],
                "raw_text": raw_text,
                "status": "ok",
            }

        except Exception as e:
            last_error = e
            print(f"Gemini warning on question: {question_text}")
            print(f"Attempt {attempt+1}/{max_retries} failed: {e}")

            if attempt < max_retries - 1:
                time.sleep(10 * (attempt + 1))   # 10 sec, then 20 sec, then 30 sec

    return {
        "answer": "not_sure",
        "reason": f"Fallback after Gemini failure: {last_error}",
        "raw_text": "",
        "status": "api_error",
    }

In [12]:
# ------------------------------------------------------------
# Support tier based on age
# ------------------------------------------------------------
def get_support_tier(state: Dict[str, Any], category_key: str) -> str:
    child = state["child"]
    chrono = min(child["chronological_months"], 60)
    dev_age = state["dev_age"].get(category_key, chrono)
    delay_est = state["delay_estimates"].get(category_key, {}).get("delay_months", 0)

    gap = max(0, chrono - dev_age)

    light_gap_threshold = max(2, round(0.10 * chrono))
    primary_gap_threshold = max(5, round(0.20 * chrono))

    light_delay_threshold = max(4, round(0.10 * chrono))
    primary_delay_threshold = max(6, round(0.20 * chrono))

    support_score = (
        0.65 * (gap / primary_gap_threshold) +
        0.35 * (delay_est / primary_delay_threshold)
    )

    if gap >= primary_gap_threshold or support_score >= 1.0:
        return "needs_special_support"

    if (
        (gap >= light_gap_threshold and delay_est >= light_delay_threshold)
        or support_score >= 0.60
    ):
        return "monitor_and_enrich"

    return "no_special_support"

In [13]:
# ------------------------------------------------------------
# Shared planning engine
# ------------------------------------------------------------
def no_special_support_needed(state: Dict[str, Any], category_key: str) -> bool:
    return get_support_tier(state, category_key) == "no_special_support"

def select_next_milestones(state: Dict[str, Any], category_key: str, max_milestones: int = 6) -> Dict[str, Any]:
    child = state["child"]
    dev_age = state["dev_age"].get(category_key, None)

    if dev_age is None:
        raise ValueError(f"No developmental age found for {category_key}. Run Q&A first.")

    if no_special_support_needed(state, category_key):
        return {
            "status": "no_special_support",
            "message": f"We do not think {child['name']} has a meaningful delay in {DOMAIN_CONFIG[category_key]['display']} and may not need special support in this category right now.",
            "milestones": [],
        }

    min_m = max(2, dev_age)
    max_m = min(60, dev_age + 12)

    subset = get_category_questions(category_key, min_months=min_m, max_months=max_m)

    if subset.empty:
        return {
            "status": "success",
            "milestones": [],
        }

    subset = subset.sort_values(["months", "milestone"]).copy()

    milestones = []
    seen = set()

    for _, row in subset.iterrows():
        key = (int(row["months"]), str(row["milestone"]).strip())
        if key in seen:
            continue
        seen.add(key)

        milestones.append({
            "months": int(row["months"]),
            "milestone": row["milestone"],
        })

    return {
        "status": "success",
        "milestones": milestones[:max_milestones],
    }

def generate_category_activity_bank(
    state: Dict[str, Any],
    category_key: str,
    model: str = "gpt-4o-mini",
    activities_per_category: int = 6,
) -> Dict[str, Any]:
    child = state["child"]
    category_display = DOMAIN_CONFIG[category_key]["display"]

    next_steps = select_next_milestones(state, category_key)

    if next_steps["status"] == "no_special_support":
        state["activity_banks"][category_key] = {
            "status": "no_special_support",
            "summary": next_steps["message"],
            "activities": [],
        }
        return state["activity_banks"][category_key]

    dev_age = state["dev_age"][category_key]
    chrono_months = min(child["chronological_months"], 60)
    milestone_gap = max(0, chrono_months - dev_age)

    milestone_lines = "\n".join(
        [f"- ({m['months']} months) {m['milestone']}" for m in next_steps["milestones"]]
    )
    if not milestone_lines:
        milestone_lines = "- No specific milestone items available in this range."

    if OpenAI is None or not os.environ.get("OPENAI_API_KEY"):
        fallback_activities = []
        for i, m in enumerate(next_steps["milestones"][:activities_per_category], start=1):
            fallback_activities.append({
                "activity_id": f"{category_key}_{i}",
                "title": f"Practice: {m['milestone']}",
                "instructions": f"Use a short, play-based home activity to practice: {m['milestone']}.",
                "duration_min": 5,
                "materials": "common toys or household items",
                "level": "current_or_next",
                "category": category_display,
                "is_extended_activity": False,
                "extended_reason": "",
            })

        while len(fallback_activities) < activities_per_category:
            i = len(fallback_activities) + 1
            fallback_activities.append({
                "activity_id": f"{category_key}_{i}",
                "title": f"{category_display} practice {i}",
                "instructions": f"Do one short parent-guided home activity that supports {category_display.lower()} skills.",
                "duration_min": 5,
                "materials": "common toys or household items",
                "level": "current_or_next",
                "category": category_display,
                "is_extended_activity": False,
                "extended_reason": "",
            })

        state["activity_banks"][category_key] = {
            "status": "fallback",
            "summary": f"Fallback activity bank used for {category_display}.",
            "activities": fallback_activities,
        }
        return state["activity_banks"][category_key]

    client = OpenAI()

    prompt = f"""
You are a pediatric home-support planning agent helping a parent at home.

This is NOT a diagnosis and NOT a formal treatment plan.
Create a CATEGORY ACTIVITY BANK, not a day-by-day schedule.

Child:
- Name: {child['name']}
- Chronological age: {child['chronological_months']} months
- Diagnosis / condition: {child['diagnosis']}
- Parent concern: {child['concern']}
- Category: {category_display}
- Estimated developmental age in this category: {dev_age} months
- Estimated milestone gap in this category: {milestone_gap} months

Relevant milestone targets:
{milestone_lines}

Task:
Create {activities_per_category} realistic home activities for this category.

Instructions:
1. Activities must fit the child's chronological age and estimated developmental level.
2. Activities should be practical for home.
3. Keep language parent-friendly.
4. Include a mix of current-level practice and near next-step practice.
5. Most activities should be short and realistic for home: usually 3, 5, 7, or 10 minutes.
6. Avoid making many 15-minute activities unless clearly justified.
7. Rare exception: some activities may naturally benefit from longer continuous time, such as a short playdate, playground peer practice, or group social activity.
8. Only mark an activity as extended if it is clearly justified.
9. Extended activities should usually be 15 or 20 minutes, not longer than 20 unless parents have more than 20 minutes available per day.
10. Do not make more than 1 or 2 extended activities in this category.
11. Return strict JSON only.

Required JSON:
{{
  "summary": "...",
  "activities": [
    {{
      "activity_id": "1",
      "title": "...",
      "instructions": "...",
      "duration_min": 5,
      "materials": "...",
      "level": "current_or_next",
      "category": "<category name>",
      "is_extended_activity": false,
      "extended_reason": ""
    }}
  ]
}}
""".strip()

    try:
        resp = client.chat.completions.create(
            model=model,
            temperature=0.3,
            response_format={"type": "json_object"},
            messages=[
                {
                    "role": "system",
                    "content": "You return strict JSON only and stay non-diagnostic, practical, and parent-friendly."
                },
                {"role": "user", "content": prompt},
            ],
        )

        bank = json.loads(resp.choices[0].message.content)

        if "activities" not in bank or not isinstance(bank["activities"], list):
            bank["activities"] = []

        for idx, activity in enumerate(bank["activities"], start=1):
            activity.setdefault("activity_id", f"{category_key}_{idx}")
            activity.setdefault("category", category_display)
            activity.setdefault("duration_min", 5)
            activity.setdefault("materials", "common household items")
            activity.setdefault("level", "current_or_next")
            activity.setdefault("is_extended_activity", False)
            activity.setdefault("extended_reason", "")

        state["activity_banks"][category_key] = {
            "status": "success",
            "summary": bank.get("summary", f"Created activity bank for {category_display}."),
            "activities": bank["activities"][:activities_per_category],
        }
        return state["activity_banks"][category_key]

    except Exception as e:
        fallback_activities = []
        for i, m in enumerate(next_steps["milestones"][:activities_per_category], start=1):
            fallback_activities.append({
                "activity_id": f"{category_key}_{i}",
                "title": f"Practice: {m['milestone']}",
                "instructions": f"Use a short, play-based home activity to practice: {m['milestone']}.",
                "duration_min": 5,
                "materials": "common toys or household items",
                "level": "current_or_next",
                "category": category_display,
                "is_extended_activity": False,
                "extended_reason": "",
            })

        while len(fallback_activities) < activities_per_category:
            i = len(fallback_activities) + 1
            fallback_activities.append({
                "activity_id": f"{category_key}_{i}",
                "title": f"{category_display} practice {i}",
                "instructions": f"Do one short parent-guided home activity that supports {category_display.lower()} skills.",
                "duration_min": 5,
                "materials": "common toys or household items",
                "level": "current_or_next",
                "category": category_display,
                "is_extended_activity": False,
                "extended_reason": "",
            })

        state["activity_banks"][category_key] = {
            "status": "fallback",
            "summary": f"Fallback activity bank used for {category_display} because OpenAI failed: {e}",
            "activities": fallback_activities,
        }
        return state["activity_banks"][category_key]

def allocate_weekly_slots(state: Dict[str, Any]) -> Dict[str, Any]:
    child = state["child"]

    if "daily_time_min" not in state["child"]:
        raise ValueError(
            "Missing daily_time_min in state['child']. "
            "Check intake_agent_live() and make sure it stores daily_time_min."
        )

    chrono = min(child["chronological_months"], 60)
    daily_time_min = int(child["daily_time_min"])
    weekly_minutes = daily_time_min * 5

    supported_categories = []
    gap_by_category = {}
    weight_by_category = {}

    for category_key in DOMAIN_CONFIG:
        tier = get_support_tier(state, category_key)
        if tier == "no_special_support":
            continue

        supported_categories.append(category_key)

        dev_age = state["dev_age"][category_key]
        gap = max(0, chrono - dev_age)
        gap_by_category[category_key] = gap

        if tier == "needs_special_support":
            weight_by_category[category_key] = max(1, gap)
        else:  # monitor_and_enrich
            weight_by_category[category_key] = max(1, gap) * 0.5

    if not supported_categories:
        allocation = {
            "daily_time_min": daily_time_min,
            "weekly_minutes": weekly_minutes,
            "supported_categories": [],
            "gap_by_category": {},
            "target_minutes_by_category": {},
        }
        state["weekly_slot_allocation"] = allocation
        return allocation

    base_minutes_per_category = max(5, daily_time_min // 2)

    target_minutes_by_category = {
        k: base_minutes_per_category for k in supported_categories
    }

    used_minutes = base_minutes_per_category * len(supported_categories)
    remaining_minutes = max(0, weekly_minutes - used_minutes)

    weights = weight_by_category.copy()    total_weight = sum(weights.values())

    if total_weight > 0 and remaining_minutes > 0:
        for k in supported_categories:
            extra = round(remaining_minutes * (weights[k] / total_weight))
            target_minutes_by_category[k] += extra

    total_target = sum(target_minutes_by_category.values())
    while total_target > weekly_minutes:
        biggest = max(target_minutes_by_category, key=target_minutes_by_category.get)
        if target_minutes_by_category[biggest] > 5:
            target_minutes_by_category[biggest] -= 1
            total_target -= 1
        else:
            break

    allocation = {
        "daily_time_min": daily_time_min,
        "weekly_minutes": weekly_minutes,
        "supported_categories": supported_categories,
        "gap_by_category": gap_by_category,
        "target_minutes_by_category": target_minutes_by_category,
    }

    state["weekly_slot_allocation"] = allocation
    return allocation

def pick_activity_that_fits(
    activities: List[Dict[str, Any]],
    used_indices: set,
    remaining_minutes: int,
) -> Optional[Dict[str, Any]]:
    candidates = []

    for idx, activity in enumerate(activities):
        if idx in used_indices:
            continue
        duration = int(activity.get("duration_min", 5))
        if duration <= remaining_minutes:
            candidates.append((duration, idx, activity))

    if not candidates:
        return None

    candidates = sorted(candidates, key=lambda x: x[0])
    duration, idx, activity = candidates[0]
    used_indices.add(idx)
    return activity

def pick_weekly_bonus_extended_activity(
    state: Dict[str, Any],
    extended_in_schedule_threshold: int = 15,
    bonus_extended_cap_min: int = 20,
) -> Optional[Dict[str, Any]]:
    daily_time_min = int(state["child"]["daily_time_min"])

    if daily_time_min >= extended_in_schedule_threshold:
        return None

    allocation = state.get("weekly_slot_allocation", {})
    gap_by_category = allocation.get("gap_by_category", {})

    candidate_categories = sorted(
        gap_by_category.keys(),
        key=lambda k: gap_by_category[k],
        reverse=True,
    )

    for category_key in candidate_categories:
        bank = state.get("activity_banks", {}).get(category_key, {})
        activities = bank.get("activities", [])

        for activity in activities:
            if not activity.get("is_extended_activity", False):
                continue

            duration = int(activity.get("duration_min", 5))
            if duration > bonus_extended_cap_min:
                continue

            return {
                "category_key": category_key,
                "category": DOMAIN_CONFIG[category_key]["display"],
                "title": activity.get("title"),
                "instructions": activity.get("instructions"),
                "duration_min": duration,
                "materials": activity.get("materials", "common household items"),
                "level": activity.get("level", "current_or_next"),
                "extended_reason": activity.get("extended_reason", ""),
            }

    return None

def build_weekly_schedule(state: Dict[str, Any]) -> Dict[str, Any]:
    if "weekly_slot_allocation" not in state:
        allocate_weekly_slots(state)

    allocation = state["weekly_slot_allocation"]
    daily_time_min = allocation["daily_time_min"]
    target_minutes_by_category = allocation["target_minutes_by_category"]

    days = {
        "day_1": {"items": [], "total_minutes": 0},
        "day_2": {"items": [], "total_minutes": 0},
        "day_3": {"items": [], "total_minutes": 0},
        "day_4": {"items": [], "total_minutes": 0},
        "day_5": {"items": [], "total_minutes": 0},
    }

    if not target_minutes_by_category:
        state["weekly_schedule"] = {
            "status": "no_special_support",
            "summary": "No categories need a scheduled weekly activity plan right now.",
            "days": days,
        }
        return state["weekly_schedule"]

    assigned_minutes_by_category = {k: 0 for k in target_minutes_by_category.keys()}
    used_activity_indices = {k: set() for k in target_minutes_by_category.keys()}
    day_names = list(days.keys())

    progress_made = True
    while progress_made:
        progress_made = False

        categories_in_priority_order = sorted(
            target_minutes_by_category.keys(),
            key=lambda k: target_minutes_by_category[k] - assigned_minutes_by_category[k],
            reverse=True,
        )

        for day_name in day_names:
            remaining_day_minutes = daily_time_min - days[day_name]["total_minutes"]
            if remaining_day_minutes <= 0:
                continue

            for category_key in categories_in_priority_order:
                remaining_cat_minutes = (
                    target_minutes_by_category[category_key] - assigned_minutes_by_category[category_key]
                )
                if remaining_cat_minutes <= 0:
                    continue

                bank = state["activity_banks"].get(category_key, {})
                activities = bank.get("activities", [])

                if not activities:
                    continue

                activity = pick_activity_that_fits(
                    activities=activities,
                    used_indices=used_activity_indices[category_key],
                    remaining_minutes=remaining_day_minutes,
                )

                if activity is None:
                    continue

                duration = int(activity.get("duration_min", 5))

                days[day_name]["items"].append({
                    "category_key": category_key,
                    "category": DOMAIN_CONFIG[category_key]["display"],
                    "title": activity.get("title"),
                    "instructions": activity.get("instructions"),
                    "duration_min": duration,
                    "materials": activity.get("materials", "common household items"),
                    "level": activity.get("level", "current_or_next"),
                })

                days[day_name]["total_minutes"] += duration
                assigned_minutes_by_category[category_key] += duration
                progress_made = True
                break

    bonus_activity = pick_weekly_bonus_extended_activity(
        state,
        extended_in_schedule_threshold=15,
        bonus_extended_cap_min=20,
    )

    schedule = {
        "status": "success",
        "summary": "Weekly schedule built from category activity banks and true daily minute limits.",
        "daily_time_min": daily_time_min,
        "target_minutes_by_category": target_minutes_by_category,
        "assigned_minutes_by_category": assigned_minutes_by_category,
        "days": days,
        "weekly_bonus_activity": bonus_activity,
    }

    state["weekly_schedule"] = schedule
    return schedule

def summarizer_agent(state: Dict[str, Any]) -> pd.DataFrame:
    child = state["child"]
    rows = []

    allocation = state.get("weekly_slot_allocation", {}).get("target_minutes_by_category", {})

    for category_key in DOMAIN_CONFIG:
        delay_info = state["delay_estimates"].get(category_key, {})
        dev_age = state["dev_age"].get(category_key)

        chrono_for_gap = min(child.get("chronological_months", 0), 60)
        milestone_gap = None if dev_age is None else max(0, chrono_for_gap - dev_age)

        bank = state.get("activity_banks", {}).get(category_key, {})

        rows.append({
            "child_name": child.get("name"),
            "chronological_age_months": child.get("chronological_months"),
            "diagnosis": child.get("diagnosis"),
            "concern": child.get("concern"),
            "daily_time_min": child.get("daily_time_min"),
            "category": DOMAIN_CONFIG[category_key]["display"],
            "estimated_delay_months": delay_info.get("delay_months"),
            "delay_reason": delay_info.get("reason", ""),
            "estimated_dev_age_months": dev_age,
            "milestone_gap_months": milestone_gap,
            "support_tier": get_support_tier(state, category_key),
            "needs_special_support": not no_special_support_needed(state, category_key),
            "weekly_target_minutes_for_category": allocation.get(category_key, 0),
            "activity_bank_status": bank.get("status", "missing"),
            "activity_bank_summary": bank.get("summary", ""),
        })

    return pd.DataFrame(rows)

def display_weekly_schedule(state: Dict[str, Any]) -> None:
    schedule = state.get("weekly_schedule", {})

    print_banner("WEEKLY SCHEDULE")

    if schedule.get("status") == "no_special_support":
        print(schedule.get("summary", "No weekly schedule available."))
        return

    print("Summary:", schedule.get("summary", ""))
    print("Daily time budget:", schedule.get("daily_time_min"))
    print("Target minutes by category:", schedule.get("target_minutes_by_category", {}))
    print("Assigned minutes by category:", schedule.get("assigned_minutes_by_category", {}))

    for day_name, day_info in schedule.get("days", {}).items():
        print("\n" + "-" * 100)
        print(f"{day_name.upper()} — total {day_info.get('total_minutes', 0)} min")
        print("-" * 100)

        items = day_info.get("items", [])
        if not items:
            print("No scheduled activities.")
            continue

        for item in items:
            print(f"- [{item['category']}] {item['title']} ({item['duration_min']} min)")
            print(f"  Instructions: {item['instructions']}")
            print(f"  Materials: {item['materials']}")

    bonus = schedule.get("weekly_bonus_activity")
    if bonus:
        print("\n" + "=" * 100)
        print("OPTIONAL WEEKLY BONUS ACTIVITY")
        print("=" * 100)
        print(f"- [{bonus['category']}] {bonus['title']} ({bonus['duration_min']} min)")
        print(f"  Instructions: {bonus['instructions']}")
        print(f"  Materials: {bonus['materials']}")
        print(f"  Why extended: {bonus.get('extended_reason', '')}")

def run_downstream_planning(state: Dict[str, Any], display_schedule: bool = True) -> pd.DataFrame:
    for category_key in DOMAIN_CONFIG:
        generate_category_activity_bank(state, category_key)

    allocate_weekly_slots(state)
    build_weekly_schedule(state)

    summary_df = summarizer_agent(state)

    if display_schedule:
        display_weekly_schedule(state)

    return summary_df

SyntaxError: invalid syntax (3098030331.py, line 297)

In [14]:
# ------------------------------------------------------------
# Main runners
# ------------------------------------------------------------
def run_all_categories_live():
    state = new_state()
    intake_agent_live(state)
    print_child_reference(state)

    delay_agent_all_categories(state)

    for category_key in DOMAIN_CONFIG:
        qna_agent_live(
            state,
            category_key,
            min_yes_confirm=2,
            yes_ratio_confirm=0.70,
        )

    summary_df = run_downstream_planning(state, display_schedule=True)
    return state, summary_df

def qna_agent_gemini_parent(
    state: Dict[str, Any],
    category_key: str,
    ask_limit_per_band: int = 3,
    min_yes_confirm: int = 2,
    yes_ratio_confirm: float = 0.60,
    gemini_model: str = "gemini-2.5-flash",
    delay_between_questions_sec: int = 8,
) -> Dict[str, Any]:
    print_banner(f"MILESTONE Q&A AGENT — {DOMAIN_CONFIG[category_key]['display']}")

    if "qna" not in state:
        state["qna"] = {}
    if "dev_age" not in state:
        state["dev_age"] = {}

    questions = build_milestone_questions(
        state,
        category_key,
        window_months=24,
        target_questions_per_band=3,
        max_bands=3,
        max_questions_total=9,
    )

    if not questions:
        raise ValueError(
            f"No milestone questions found for category_key={category_key}. "
            f"Check category mapping and CDC normalization."
        )

    asked = []
    prior_answers_for_gemini = []

    questions_by_month = {}
    for q in questions:
        questions_by_month.setdefault(q["months"], []).append(q)

    for month in sorted(questions_by_month.keys()):
        print(f"\n--- Asking {DOMAIN_CONFIG[category_key]['display']} milestones around {month} months ---")

        for q in questions_by_month[month][:ask_limit_per_band]:
            sim = gemini_parent_answer_one_question(
                child=state["child"],
                category_display=DOMAIN_CONFIG[category_key]["display"],
                question_text=q["question_text"],
                prior_answers=prior_answers_for_gemini,
                model=gemini_model,
                max_retries=3,
            )

            norm = normalize_answer(sim["answer"])

            asked_item = {
                "question_id": q["question_id"],
                "months": q["months"],
                "milestone": q["milestone"],
                "raw_answer": sim["answer"],
                "norm_answer": norm,
                "score": ANSWER_SCORES[norm],
                "parent_reason": sim["reason"],
                "answer_status": sim.get("status", "ok"),
            }
            asked.append(asked_item)

            prior_answers_for_gemini.append({
                "question": q["question_text"],
                "answer": sim["answer"],
            })

            print(f"Q: {q['question_text']}")
            print(
                f"A (Gemini-parent): {sim['answer']} | "
                f"status: {sim.get('status', 'ok')} | "
                f"reason: {sim['reason']}"
            )

            time.sleep(delay_between_questions_sec)

    usable_answers = [a for a in asked if a.get("answer_status") != "api_error"]

    dev_age = compute_dev_age_from_answers(
        usable_answers,
        min_yes_confirm=min_yes_confirm,
        yes_ratio_confirm=yes_ratio_confirm,
    )

    state["qna"][category_key] = asked
    state["dev_age"][category_key] = dev_age

    chrono = state["child"]["chronological_months"]

    print("\nQuestion / answer transcript:")
    for item in asked:
        print(
            f"- [{item['months']}m] {item['milestone']} -> {item['norm_answer']} "
            f"| status: {item.get('answer_status', 'ok')} "
            f"| reason: {item.get('parent_reason', '')}"
        )

    band_summary = summarize_answers_by_band(usable_answers)
    print("\nBand summary:")
    for month, info in band_summary.items():
        print(
            f"  {month}m | total={info['total']} | yes={info['yes']} | "
            f"partial={info['partial']} | no={info['no']} | yes_ratio={info['yes_ratio']}"
        )

    print(
        f"\nEstimated developmental age for {DOMAIN_CONFIG[category_key]['display']}: "
        f"{dev_age} months (chronological age {chrono} months)"
    )

    return {
        "category": category_key,
        "dev_age_months": dev_age,
        "answers": asked,
        "band_summary": band_summary,
    }

def run_downstream_planning(state: Dict[str, Any], display_schedule: bool = True) -> pd.DataFrame:
    for category_key in DOMAIN_CONFIG:
        generate_category_activity_bank(state, category_key)

    allocate_weekly_slots(state)
    build_weekly_schedule(state)

    summary_df = summarizer_agent(state)

    if display_schedule:
        display_weekly_schedule(state)

    return summary_df

def run_all_categories_gemini_parent(case: Dict[str, Any], daily_time_min: int = 10, gemini_model: str = "gemini-2.5-flash"):
    state = init_state_from_case(case, daily_time_min=daily_time_min)
    print_child_reference(state)

    delay_agent_all_categories(state)

    for category_key in DOMAIN_CONFIG:
        # qna_agent_gemini_parent(
        #     state,
        #     category_key,
        #     ask_limit_per_band=5,
        #     min_yes_confirm=2,
        #     yes_ratio_confirm=0.70,
        #     gemini_model=gemini_model,
        # )
        qna_agent_gemini_parent(
            state,
            category_key,
            ask_limit_per_band=3,
            min_yes_confirm=2,
            yes_ratio_confirm=0.60,
            gemini_model=gemini_model,
            delay_between_questions_sec=10,
        )

    summary_df = run_downstream_planning(state, display_schedule=True)
    return state, summary_df

def print_case_reference(case: dict, daily_time_min: int):
    print("\n" + "=" * 110)
    print(f"STARTING {case['case_id']} | {case['child_name']} | {case['age_label']} | {case['diagnosis']}")
    print("=" * 110)
    print(f"Concerns: {case['concerns']}")
    print(f"Daily time: {daily_time_min} min/day")

def run_case_live_from_prefilled_case(
    case: dict,
    default_daily_time_min: int = 10,
    min_yes_confirm: int = 2,
    yes_ratio_confirm: float = 0.70,
):
    raw_daily = input(
        f"[{case['case_id']}] Daily minutes available? Press Enter for default {default_daily_time_min}: "
    ).strip()
    daily_time_min = int(raw_daily) if raw_daily else int(default_daily_time_min)

    state = init_state_from_case(case, daily_time_min=daily_time_min)
    print_case_reference(case, daily_time_min)
    delay_agent_all_categories(state)

    for category_key in DOMAIN_CONFIG:
        qna_agent_live(
            state,
            category_key,
            min_yes_confirm=min_yes_confirm,
            yes_ratio_confirm=yes_ratio_confirm,
        )

    summary_df = run_downstream_planning(state, display_schedule=True)
    return state, summary_df

def run_case_gemini_from_prefilled_case(
    case: dict,
    daily_time_min: int = 10,
    min_yes_confirm: int = 2,
    yes_ratio_confirm: float = 0.70,
    gemini_model: str = "gemini-2.5-flash",
):
    state = init_state_from_case(case, daily_time_min=daily_time_min)
    print_case_reference(case, daily_time_min)
    delay_agent_all_categories(state)

    for category_key in DOMAIN_CONFIG:
        qna_agent_gemini_parent(
            state,
            category_key,
            ask_limit_per_band=5,
            min_yes_confirm=min_yes_confirm,
            yes_ratio_confirm=yes_ratio_confirm,
            gemini_model=gemini_model,
        )

    summary_df = run_downstream_planning(state, display_schedule=True)
    return state, summary_df

In [15]:
# ------------------------------------------------------------
# Test cases
# ------------------------------------------------------------
cases = [
    {"case_id": "C01", "child_name": "Noah", "age_months": 10, "age_label": "10 months",
     "diagnosis": "Down syndrome",
     "concerns": "Not sitting independently yet, Hypotonia, slow feeding, socially engaged"},

    {"case_id": "C02", "child_name": "Maya", "age_months": 18, "age_label": "18 months",
     "diagnosis": "No formal diagnosis",
     "concerns": "No words yet, uses gestures, understands simple commands, good eye contact"},

    {"case_id": "C03", "child_name": "Leo", "age_months": 26, "age_label": "2y 2m",
     "diagnosis": "Autism spectrum disorder",
     "concerns": "Limited eye contact and few words, Repetitive play, transitions hard, picky eating"},

    {"case_id": "C04", "child_name": "Sofia", "age_months": 37, "age_label": "3y 1m",
     "diagnosis": "Prematurity history",
     "concerns": "Speech hard to understand, Fine motor delay, good comprehension"},

    {"case_id": "C05", "child_name": "Ethan", "age_months": 54, "age_label": "4y 6m",
     "diagnosis": "Cerebral palsy",
     "concerns": "Trouble keeping up in preschool, Self-care delays, frequent falls, difficulty with stairs, uses walker sometimes, good cognition"},

    {"case_id": "C06", "child_name": "Amina", "age_months": 60, "age_label": "5 years",
     "diagnosis": "Global developmental delay",
     "concerns": "Learning and routines are hard, Short attention span, follows only simple directions"},

    {"case_id": "C07", "child_name": "Lucas", "age_months": 14, "age_label": "14 months",
     "diagnosis": "Fragile X syndrome",
     "concerns": "Not walking, limited babbling, Sensory sensitivity, shy socially"},

    {"case_id": "C08", "child_name": "Emma", "age_months": 32, "age_label": "2y 8m",
     "diagnosis": "Suspected childhood apraxia of speech",
     "concerns": "Very limited speech output, Uses gestures, gets frustrated, inconsistent sounds"},

    {"case_id": "C09", "child_name": "Zain", "age_months": 48, "age_label": "4 years",
     "diagnosis": "Attention and self-regulation concerns (possible ADHD)",
     "concerns": "Very active, impulsive, Big emotional reactions, interrupts, difficulty following routines, social interest intact"},

    {"case_id": "C10", "child_name": "Isla", "age_months": 44, "age_label": "3y 8m",
     "diagnosis": "SYNGAP1-related disorder",
     "concerns": "Delayed speech and balance, Attention issues, clumsy gait, limited pretend play"},

    {"case_id": "C11", "child_name": "Aria", "age_months": 20, "age_label": "20 months",
     "diagnosis": "No diagnosis",
     "concerns": "Only ~10 words, very active, good eye contact, understands well"},

    {"case_id": "C12", "child_name": "Mateo", "age_months": 30, "age_label": "2y 6m",
     "diagnosis": "No formal diagnosis",
     "concerns": "Good eye contact but no words, lines up toys, responds to name inconsistently"},

    {"case_id": "C13", "child_name": "Luna", "age_months": 42, "age_label": "3y 6m",
     "diagnosis": "Speech delay",
     "concerns": "Limited speech, understands well, parents have very limited time for activities"},

    {"case_id": "C14", "child_name": "Ryan", "age_months": 50, "age_label": "4y 2m",
     "diagnosis": "No diagnosis",
     "concerns": "Tantrums, difficulty with transitions, strong language skills, social but rigid"},
]

df_cases = pd.DataFrame(cases)
display(df_cases)

,case_id,child_name,age_months,age_label,diagnosis,concerns
0,C01,Noah,10,10 months,Down syndrome,"Not sitting independently yet, Hypotonia, slow..."
1,C02,Maya,18,18 months,No formal diagnosis,"No words yet, uses gestures, understands simpl..."
2,C03,Leo,26,2y 2m,Autism spectrum disorder,"Limited eye contact and few words, Repetitive ..."
3,C04,Sofia,37,3y 1m,Prematurity history,"Speech hard to understand, Fine motor delay, g..."
4,C05,Ethan,54,4y 6m,Cerebral palsy,"Trouble keeping up in preschool, Self-care del..."
5,C06,Amina,60,5 years,Global developmental delay,"Learning and routines are hard, Short attentio..."
6,C07,Lucas,14,14 months,Fragile X syndrome,"Not walking, limited babbling, Sensory sensiti..."
7,C08,Emma,32,2y 8m,Suspected childhood apraxia of speech,"Very limited speech output, Uses gestures, get..."
8,C09,Zain,48,4 years,Attention and self-regulation concerns (possib...,"Very active, impulsive, Big emotional reaction..."
9,C10,Isla,44,3y 8m,SYNGAP1-related disorder,"Delayed speech and balance, Attention issues, ..."


In [16]:
# ------------------------------------------------------------
# Advisor-report helpers
# ------------------------------------------------------------
def severity_from_gap(gap_months: float) -> str:
    gap_months = 0 if pd.isna(gap_months) else float(gap_months)
    if gap_months <= 6:
        return "minimal / mild"
    elif gap_months <= 18:
        return "moderate"
    else:
        return "significant"

def split_concerns_to_bullets(concern_text: str, max_items: int = 4):
    items = [x.strip(" -•\n\t") for x in re.split(r"[,\n;]+", str(concern_text)) if x.strip()]
    return items[:max_items]

def extract_parent_qa_lines(state: dict, max_lines_for_pdf: int = 12):
    full_lines = []
    for category_key in DOMAIN_CONFIG.keys():
        answers = state.get("qna", {}).get(category_key, [])
        for a in answers:
            milestone = str(a.get("milestone", "")).strip()
            norm_answer = str(a.get("norm_answer", a.get("raw_answer", ""))).strip()
            months = a.get("months", "")
            full_lines.append(f"[{months}m] {milestone} -> {norm_answer}")

    pdf_lines = full_lines[:max_lines_for_pdf]
    return full_lines, pdf_lines

def extract_focus_areas(summary_df: pd.DataFrame, max_items: int = 3):
    if summary_df is None or summary_df.empty:
        return []

    work = summary_df.copy()
    if "needs_special_support" in work.columns:
        work = work[work["needs_special_support"] == True].copy()

    if work.empty:
        return []

    sort_col = "milestone_gap_months" if "milestone_gap_months" in work.columns else "estimated_delay_months"
    work = work.sort_values(sort_col, ascending=False)
    return work["category"].tolist()[:max_items]

def extract_activities_for_case(state: dict, max_activities: int = 5):
    activities = []

    weekly_schedule = state.get("weekly_schedule", {})
    days = weekly_schedule.get("days", {})

    if isinstance(days, dict):
        for day_label, day_info in days.items():
            day_items = day_info.get("items", [])
            for item in day_items:
                activities.append({
                    "day": day_label,
                    "category": item.get("category", item.get("category_display", "")),
                    "title": item.get("title", ""),
                    "description": item.get("instructions", item.get("description", "")),
                    "duration_min": item.get("duration_min", ""),
                    "goal": item.get("goal", item.get("level", item.get("category", ""))),
                })

    if not activities:
        for category_key, bank in state.get("activity_banks", {}).items():
            for item in bank.get("activities", [])[:2]:
                activities.append({
                    "day": "",
                    "category": DOMAIN_CONFIG.get(category_key, {}).get("display", category_key),
                    "title": item.get("title", ""),
                    "description": item.get("instructions", item.get("description", "")),
                    "duration_min": item.get("duration_min", ""),
                    "goal": item.get("goal", item.get("level", "current_or_next")),
                })

    return activities[:max_activities]

def build_case_payload(case: dict, state: dict, summary_df: pd.DataFrame):
    concerns_bullets = split_concerns_to_bullets(case.get("concerns", ""), max_items=4)
    parent_qa_full, parent_qa_pdf = extract_parent_qa_lines(state, max_lines_for_pdf=12)
    focus_areas = extract_focus_areas(summary_df, max_items=3)
    activities = extract_activities_for_case(state, max_activities=5)

    domain_rows = []
    if summary_df is not None and not summary_df.empty:
        for _, row in summary_df.iterrows():
            gap = row.get("milestone_gap_months", row.get("estimated_delay_months", None))
            domain_rows.append({
                "category": row.get("category", ""),
                "estimated_dev_age_months": row.get("estimated_dev_age_months", ""),
                "gap_months": gap,
                "severity": severity_from_gap(gap if gap is not None else 0),
                "needs_special_support": row.get("needs_special_support", True),
                "summary": row.get("activity_bank_summary", row.get("summary", "")),
            })

    support_categories = [r["category"] for r in domain_rows if r.get("needs_special_support", True)]
    if support_categories:
        summary_text = (
            f"This plan prioritizes {', '.join(support_categories[:3])} for {case['child_name']}. "
            f"The activities are intended to be short, parent-friendly, and aligned with the developmental areas that appeared most delayed in the interview."
        )
    else:
        summary_text = (
            f"This output did not identify a strong need for special support across the reviewed categories, "
            f"but the family can continue monitoring development and bring remaining questions to the clinician."
        )

    payload = {
        "case_id": case.get("case_id"),
        "child_name": case.get("child_name"),
        "age_label": case.get("age_label"),
        "age_months": case.get("age_months"),
        "diagnosis": case.get("diagnosis"),
        "concerns_bullets": concerns_bullets,
        "daily_time_min": state.get("child", {}).get("daily_time_min", ""),
        "parent_qa_full": parent_qa_full,
        "parent_qa_pdf": parent_qa_pdf,
        "domain_rows": domain_rows,
        "focus_areas": focus_areas,
        "activities": activities,
        "summary_text": summary_text,
    }
    return payload

def print_case_screen_summary(payload: dict):
    print("\n" + "=" * 110)
    print(f"CASE {payload['case_id']} | {payload['child_name']} | {payload['age_label']} | {payload['diagnosis']}")
    print("=" * 110)

    print("\n1. CHILD PROFILE")
    print(f"- Name: {payload['child_name']}")
    print(f"- Age: {payload['age_label']} ({payload['age_months']} months)")
    print(f"- Diagnosis: {payload['diagnosis']}")
    print(f"- Daily time available: {payload['daily_time_min']} min/day")
    print("- Key concerns:")
    for c in payload["concerns_bullets"]:
        print(f"  • {c}")

    print("\n2. PARENT INPUT (full transcript)")
    for line in payload["parent_qa_full"]:
        print(f"- {line}")

    print("\n3. GENEX OUTPUT")
    domain_df = pd.DataFrame(payload["domain_rows"])
    if not domain_df.empty:
        display(domain_df[["category", "estimated_dev_age_months", "gap_months", "severity", "needs_special_support"]])

    print("\nFocus areas:")
    for x in payload["focus_areas"]:
        print(f"  • {x}")

    print("\nDaily activities:")
    for i, a in enumerate(payload["activities"], start=1):
        print(f"{i}. {a['title']} | {a['category']} | {a['duration_min']} min")
        print(f"   Goal: {a['goal']}")
        print(f"   Description: {a['description']}")

    print("\n4. SUMMARY")
    print(payload["summary_text"])

    print("\n5. ADVISOR REVIEW SECTION")
    print("Rate 1–5:")
    print("- Clinical appropriateness")
    print("- Safety")
    print("- Practicality for parents")
    print("- Clarity")
    print("- Overall usefulness")
    print("Short feedback:")
    print("- What would you change?")
    print("- What is missing?")
    print("- Any concerns?")

In [17]:
# ------------------------------------------------------------
# PDF rendering + rating sheet
# ------------------------------------------------------------
def wrap_text_to_width(text, font_name, font_size, max_width):
    words = str(text).split()
    if not words:
        return [""]

    lines = []
    current = words[0]
    for word in words[1:]:
        trial = current + " " + word
        if stringWidth(trial, font_name, font_size) <= max_width:
            current = trial
        else:
            lines.append(current)
            current = word
    lines.append(current)
    return lines

def draw_wrapped_text(c, text, x, y, max_width, font_name="Helvetica", font_size=8.5, line_height=10, max_lines=None):
    lines = wrap_text_to_width(text, font_name, font_size, max_width)
    if max_lines is not None:
        original_lines = wrap_text_to_width(text, font_name, font_size, max_width)
        lines = lines[:max_lines]
        if len(lines) == max_lines and len(original_lines) > max_lines:
            lines[-1] = lines[-1][:max(0, len(lines[-1]) - 3)] + "..."
    c.setFont(font_name, font_size)
    for line in lines:
        c.drawString(x, y, line)
        y -= line_height
    return y

def draw_bullets(c, items, x, y, max_width, font_name="Helvetica", font_size=8.5, line_height=10, max_items=None):
    items = list(items)
    if max_items is not None:
        items = items[:max_items]

    for item in items:
        bullet_lines = wrap_text_to_width(str(item), font_name, font_size, max_width - 12)
        if bullet_lines:
            c.drawString(x, y, u"\u2022")
            c.drawString(x + 10, y, bullet_lines[0])
            y -= line_height
            for extra in bullet_lines[1:]:
                c.drawString(x + 10, y, extra)
                y -= line_height
        else:
            y -= line_height
    return y

def draw_box(c, x, y_top, w, h, title):
    c.roundRect(x, y_top - h, w, h, 8, stroke=1, fill=0)
    c.setFont("Helvetica-Bold", 10)
    c.drawString(x + 8, y_top - 14, title)

def render_case_page(c, payload: dict):
    page_w, page_h = landscape(LETTER)
    margin = 0.35 * inch

    left_x = margin
    mid_x = page_w / 2 + 0.10 * inch
    col_w = (page_w / 2) - (1.5 * margin)

    c.setFont("Helvetica-Bold", 15)
    c.drawString(margin, page_h - 0.45 * inch, f"{payload['case_id']} — {payload['child_name']} ({payload['age_label']})")
    c.setFont("Helvetica", 10)
    c.drawString(margin, page_h - 0.68 * inch, f"Diagnosis: {payload['diagnosis']}")

    top_y = page_h - 0.90 * inch
    profile_h = 1.60 * inch
    input_h = 2.55 * inch
    output_h = 2.20 * inch
    activity_h = 2.10 * inch
    summary_h = 0.85 * inch
    rating_h = 1.20 * inch

    draw_box(c, left_x, top_y, col_w, profile_h, "1. Child Profile")
    draw_box(c, left_x, top_y - profile_h - 0.10 * inch, col_w, input_h, "2. Parent Input (questions + answers)")
    draw_box(c, mid_x, top_y, col_w, output_h, "3. Genex Output")
    draw_box(c, mid_x, top_y - output_h - 0.10 * inch, col_w, activity_h, "4. Daily Activities")

    bottom_top = page_h - 6.65 * inch
    full_w = page_w - 2 * margin
    draw_box(c, margin, bottom_top, full_w, summary_h, "5. Summary")
    draw_box(c, margin, bottom_top - summary_h - 0.10 * inch, full_w, rating_h, "6. Advisor Review")

    y = top_y - 28
    c.setFont("Helvetica", 9)
    c.drawString(left_x + 10, y, f"Name: {payload['child_name']}")
    y -= 12
    c.drawString(left_x + 10, y, f"Age: {payload['age_label']} ({payload['age_months']} months)")
    y -= 12
    c.drawString(left_x + 10, y, f"Diagnosis: {payload['diagnosis']}")
    y -= 12
    c.drawString(left_x + 10, y, f"Daily time: {payload['daily_time_min']} min/day")
    y -= 16
    c.setFont("Helvetica-Bold", 9)
    c.drawString(left_x + 10, y, "Key concerns:")
    y -= 11
    y = draw_bullets(c, payload["concerns_bullets"], left_x + 12, y, col_w - 20, max_items=4)

    y = top_y - profile_h - 0.10 * inch - 28
    y = draw_bullets(c, payload["parent_qa_pdf"], left_x + 12, y, col_w - 20, font_size=7.8, line_height=9, max_items=12)
    c.setFont("Helvetica-Oblique", 7.5)
    c.drawString(left_x + 10, top_y - profile_h - input_h - 0.10 * inch + 10, "Full transcript is preserved in JSON output.")

    y = top_y - 28
    domain_rows = payload["domain_rows"][:4]
    c.setFont("Helvetica-Bold", 8.5)
    c.drawString(mid_x + 10, y, "Domain")
    c.drawString(mid_x + 150, y, "Dev age")
    c.drawString(mid_x + 220, y, "Gap")
    c.drawString(mid_x + 270, y, "Severity")
    y -= 11
    c.setFont("Helvetica", 8.2)
    for row in domain_rows:
        c.drawString(mid_x + 10, y, str(row["category"])[:24])
        c.drawString(mid_x + 150, y, str(row["estimated_dev_age_months"]))
        c.drawString(mid_x + 220, y, str(row["gap_months"]))
        c.drawString(mid_x + 270, y, str(row["severity"]))
        y -= 10

    y -= 8
    c.setFont("Helvetica-Bold", 9)
    c.drawString(mid_x + 10, y, "Focus areas:")
    y -= 11
    y = draw_bullets(c, payload["focus_areas"], mid_x + 12, y, col_w - 20, max_items=3)

    y = top_y - output_h - 0.10 * inch - 28
    for i, a in enumerate(payload["activities"][:5], start=1):
        line = f"{i}. {a['title']} ({a['duration_min']} min) — {a['goal']}"
        y = draw_wrapped_text(c, line, mid_x + 10, y, col_w - 20, font_name="Helvetica-Bold", font_size=8.2, line_height=9, max_lines=2)
        y = draw_wrapped_text(c, a["description"], mid_x + 16, y, col_w - 26, font_name="Helvetica", font_size=7.8, line_height=8.5, max_lines=2)
        y -= 4

    y = bottom_top - 28
    y = draw_wrapped_text(c, payload["summary_text"], margin + 10, y, full_w - 20, font_name="Helvetica", font_size=8.5, line_height=10, max_lines=4)

    y = bottom_top - summary_h - 0.10 * inch - 28
    rating_lines = [
        "Rate 1–5: Clinical appropriateness | Safety | Practicality for parents | Clarity | Overall usefulness",
        "Short feedback: What would you change? | What is missing? | Any concerns?"
    ]
    for line in rating_lines:
        y = draw_wrapped_text(c, line, margin + 10, y, full_w - 20, font_name="Helvetica", font_size=9, line_height=11, max_lines=2)

    c.showPage()

def build_advisor_rating_sheet(df_cases: pd.DataFrame) -> pd.DataFrame:
    rows = []
    for _, row in df_cases.iterrows():
        rows.append({
            "case_id": row["case_id"],
            "child_name": row["child_name"],
            "age_months": row["age_months"],
            "diagnosis": row["diagnosis"],
            "clinical_appropriateness_1to5": "",
            "safety_1to5": "",
            "practicality_for_parents_1to5": "",
            "clarity_1to5": "",
            "overall_usefulness_1to5": "",
            "what_would_you_change": "",
            "what_is_missing": "",
            "any_concerns": "",
        })
    return pd.DataFrame(rows)

In [18]:
# ------------------------------------------------------------
# Generic advisor-report export runner
# ------------------------------------------------------------
def run_cases_and_export(
    df_cases: pd.DataFrame,
    mode: str = "live",   # "live" or "gemini"
    output_dir: str = "outputs/advisor_case_pack_v2",
    default_daily_time_min: int = 10,
    pause_between_cases: bool = True,
    gemini_model: str = "gemini-2.5-flash",
):
    mode = str(mode).strip().lower()
    if mode not in {"live", "gemini"}:
        raise ValueError("mode must be 'live' or 'gemini'")

    out_dir = Path(output_dir)
    out_dir.mkdir(parents=True, exist_ok=True)

    run_stamp = datetime.now().strftime("%Y%m%d_%H%M%S")
    combined_pdf_path = out_dir / f"genex_case_report_{mode}_{run_stamp}.pdf"
    rating_csv_path = out_dir / f"advisor_rating_sheet_{mode}_{run_stamp}.csv"
    rating_xlsx_path = out_dir / f"advisor_rating_sheet_{mode}_{run_stamp}.xlsx"
    json_path = out_dir / f"case_payloads_{mode}_{run_stamp}.json"

    c = canvas.Canvas(str(combined_pdf_path), pagesize=landscape(LETTER))

    all_payloads = []
    rating_df = build_advisor_rating_sheet(df_cases)

    for i, (_, row) in enumerate(df_cases.iterrows(), start=1):
        case = row.to_dict()

        if mode == "live":
            state, summary_df = run_case_live_from_prefilled_case(
                case=case,
                default_daily_time_min=default_daily_time_min,
                min_yes_confirm=2,
                yes_ratio_confirm=0.70,
            )
        else:
            state, summary_df = run_case_gemini_from_prefilled_case(
                case=case,
                daily_time_min=default_daily_time_min,
                min_yes_confirm=2,
                yes_ratio_confirm=0.70,
                gemini_model=gemini_model,
            )

        payload = build_case_payload(case, state, summary_df)
        all_payloads.append(payload)

        print_case_screen_summary(payload)
        render_case_page(c, payload)

        per_case_summary_path = out_dir / f"{case['case_id']}_{case['child_name']}_summary.csv"
        summary_df.to_csv(per_case_summary_path, index=False)

        if pause_between_cases and i < len(df_cases):
            input("\nPress Enter to continue to the next case...")

    c.save()

    rating_df.to_csv(rating_csv_path, index=False)
    rating_df.to_excel(rating_xlsx_path, index=False)

    with open(json_path, "w", encoding="utf-8") as f:
        json.dump(all_payloads, f, indent=2)

    print("\n" + "=" * 110)
    print("DONE")
    print("=" * 110)
    print("Mode:", mode)
    print("Combined PDF:", combined_pdf_path)
    print("Rating CSV :", rating_csv_path)
    print("Rating XLSX:", rating_xlsx_path)
    print("Payload JSON:", json_path)

    return all_payloads, rating_df, combined_pdf_path

## Example run cells

Use **only one** of the cells below at a time.

### 1. Run one live case with free intake
Use this when you want to type the parent answers yourself.

### 2. Run one synthetic case
Use this when Gemini should answer as the parent from a case profile.

### 3. Run a full advisor packet export
Choose:
- `mode="live"` to answer yourself case by case
- `mode="gemini"` to batch-run synthetic parents

In [19]:
# Example 1 — one fully live run
state, summary_df = run_all_categories_live()
display(summary_df)


INTAKE AGENT

CHILD REFERENCE CHECK
Name: peter
Chronological age: 60 months
Diagnosis / condition: adhd
Parent concern: not paying attention, lack of concentration, crying, getting bullied at school
Daily time available: 10 minutes

DELAY ESTIMATOR AGENT
Movement / Physical: 0 month starting delay | There is little reason to think this domain is significantly affected.
Social / Emotional: 6 month starting delay | Social-emotional development is likely meaningfully affected due to ADHD and concerns about attention and bullying.
Language / Communication: 6 month starting delay | Language development is likely affected due to attention and concentration issues.
Cognitive / Adaptive: 12 month starting delay | Cognitive and adaptive development may be affected due to attention and concentration issues.

MILESTONE Q&A AGENT — Movement / Physical

--- Asking Movement / Physical milestones around 48 months ---

--- Asking Movement / Physical milestones around 60 months ---

Question / answer

NameError: name 'generate_category_activity_bank' is not defined

In [20]:
print("gemini_client is None:", gemini_client is None)
print("GEMINI_API_KEY visible:", bool(os.environ.get("GEMINI_API_KEY")))

gemini_client is None: False
GEMINI_API_KEY visible: True


In [21]:
print("run_downstream_planning" in globals())
print("generate_category_activity_bank" in globals())
print("allocate_weekly_slots" in globals())
print("build_weekly_schedule" in globals())
print("summarizer_agent" in globals())

True
False
False
False
False


In [22]:
# Example 2 — one synthetic case using Gemini as the parent
case = df_cases.iloc[0].to_dict()
state, summary_df = run_all_categories_gemini_parent(case, daily_time_min=10, gemini_model="gemini-2.5-flash")
display(summary_df)


CHILD REFERENCE CHECK
Name: Noah
Chronological age: 10 months
Diagnosis / condition: Down syndrome
Parent concern: Not sitting independently yet, Hypotonia, slow feeding, socially engaged
Daily time available: 10 minutes

DELAY ESTIMATOR AGENT
Movement / Physical: 6 month starting delay | Movement and physical development are likely meaningfully affected due to hypotonia and not sitting independently yet.
Social / Emotional: 6 month starting delay | Social-emotional development may be mildly affected due to the diagnosis and concerns.
Language / Communication: 6 month starting delay | Language development is likely meaningfully affected in this child.
Cognitive / Adaptive: 6 month starting delay | Cognitive and adaptive development may be mildly affected due to Down syndrome.

MILESTONE Q&A AGENT — Movement / Physical

--- Asking Movement / Physical milestones around 2 months ---
Q: Can Noah holds head up when on tummy right now? (yes / sometimes / with_help / no / not_sure)
A (Gemini

NameError: name 'generate_category_activity_bank' is not defined

In [ ]:
# # Cell for live packet
# all_payloads, advisor_rating_df, pdf_path = run_cases_and_export(
#     df_cases=df_cases,
#     mode="live",
#     output_dir="outputs/advisor_case_pack_v2",
#     default_daily_time_min=10,
#     pause_between_cases=True,
# )

# display(advisor_rating_df.head())
# pdf_path

In [ ]:
# Cell for synthetic Gemini packet
all_payloads, advisor_rating_df, pdf_path = run_cases_and_export(
    df_cases=df_cases,
    mode="gemini",
    output_dir="outputs/advisor_case_pack_v2",
    default_daily_time_min=10,
    pause_between_cases=False,
    gemini_model="gemini-2.5-flash",
)

display(advisor_rating_df.head())
pdf_path

In [ ]:
pdf_path